This is the first chapter in the implementation-dedicated part of the book. 

For some reason, the pdf I have available cuts off the last ~5 pages of this chapter, so I couldn't do the space bar implementation (21.3)

In [16]:
#Section 21.1. Plane bar element
import numpy as np

def PlaneBar2Stiffness(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS:
    OUTPUT: Ke stiffness matrix
    """

    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1
    
    Em, Gm, rho, alpha = mprop
    (A,) = fprop

    LL = x21**2+y21**2 
    L = np.sqrt(LL)
    LLL = LL*L

    Ke = (Em*A/LLL) * np.array([
        [ x21*x21,  x21*y21, -x21*x21, -x21*y21],
        [ y21*x21,  y21*y21, -y21*x21, -y21*y21],
        [-x21*x21, -x21*y21,  x21*x21,  x21*y21],
        [-y21*x21, -y21*y21,  y21*x21,  y21*y21]
    ])

    return Ke


def PlaneBar2ConsMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS:
    OUTPUT: MeC mass matrix
    """
    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop
    (A,) = fprop

    L = np.sqrt(x21**2 + y21**2)
    MeC = (rho*A*L/6) * np.array([
        [2, 0, 1, 0], [0,2,0,1], [1,0,2,0], [0, 1, 0, 2]
    ])
    return MeC


def PlaneBar2LumpMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS:
    OUTPUT: MeL lumped mass matrix
    """
    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop
    (A,) = fprop
    
    L = np.sqrt(x21**2 + y21**2)

    MeL = (rho * A * L / 2) * np.identity(4)
    return MeL

In [20]:
# Testing of plane bar element
ncoor = [[0, 0], [30, 40]]
mprop = [1000, 0, 0.6, 0]
fprop = [5]

Ke = PlaneBar2Stiffness(ncoor, mprop, fprop)
print("Numerical Elem Stiff Matrix:")
print(Ke)
eigvals = np.linalg.eigvalsh(Ke)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of Ke =", eigvals)
print("\nSymmetry check=", np.allclose(Ke,Ke.T))
print()

MeC = PlaneBar2ConsMass(ncoor, mprop, fprop)
print("Numerical consistent Mass Matrix: ")
print(MeC)
eigvals = np.linalg.eigvalsh(MeC)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeC =", eigvals)
print("Sympetry check=", np.allclose(MeC,MeC.T))
print()

MeL = PlaneBar2LumpMass(ncoor, mprop, fprop)
print("Numerical Lumped Mass Matrix:")
print(MeL)
eigvals = np.linalg.eigvalsh(MeL)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeL", eigvals)

Numerical Elem Stiff Matrix:
[[ 36.  48. -36. -48.]
 [ 48.  64. -48. -64.]
 [-36. -48.  36.  48.]
 [-48. -64.  48.  64.]]
Eigenvalues of Ke = [  0.   0.   0. 200.]

Symmetry check= True

Numerical consistent Mass Matrix: 
[[50.  0. 25.  0.]
 [ 0. 50.  0. 25.]
 [25.  0. 50.  0.]
 [ 0. 25.  0. 50.]]
Eigenvalues of MeC = [25. 25. 75. 75.]
Sympetry check= True

Numerical Lumped Mass Matrix:
[[75.  0.  0.  0.]
 [ 0. 75.  0.  0.]
 [ 0.  0. 75.  0.]
 [ 0.  0.  0. 75.]]
Eigenvalues of MeL [75. 75. 75. 75.]


In [ ]:
# Section 21.2 Plane beam-column element

